# Setup

##  Imports

In [3]:
import os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import torch
import torchaudio
import time
import json
from concurrent.futures import ThreadPoolExecutor

# Prevent torchaudio from breaking
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None

from pyannote.audio import Pipeline
from pydub import AudioSegment

## Environment & API Setup

In [4]:
load_dotenv()
api_key = os.getenv("AQUEDUCT_API_KEY", os.getenv("API_KEY"))
hf_token = os.getenv("HF_TOKEN")

if not api_key or not hf_token:
    raise ValueError("Missing API Keys. Ensure AQUEDUCT_API_KEY and HF_TOKEN are set in .env")

client = OpenAI(api_key=api_key, base_url="https://aqueduct.ai.datalab.tuwien.ac.at/v1")

## Local Model

In [5]:
print("Loading Pyannote Diarization Model locally...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=hf_token
).to(device)

Loading Pyannote Diarization Model locally...


The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.
The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.


## Path Configuration

In [6]:
BASE_FOLDER = Path("D:/TUW-BT/user_study_recordings")

## Helper Functions

In [7]:
def transcribe_full_audio(file_path: Path) -> dict:
    """Sends the entire audio file to Whisper and requests verbose_json for timestamps."""
    print("      [API] Starting full audio transcription...")
    with open(file_path, "rb") as audio_file:
        response = client.audio.transcriptions.create(
            model="whisper-large",
            file=audio_file,
            response_format="verbose_json", # Requests timestamps
            timestamp_granularities=["segment"],
            language="en"
        )
    print("      [API] Transcription complete.")
    # The response object behaves like a dictionary/object depending on the API backend.
    # Convert to dict if it's an object
    if hasattr(response, 'model_dump'):
        return response.model_dump()
    return response

def diarize_audio_locally(file_path: Path):
    """Runs Pyannote diarization on the full audio file locally."""
    print(f"      [Local] Starting diarization on {device}...")
    full_audio = AudioSegment.from_file(str(file_path))
    mono_audio = full_audio.set_channels(1)

    samples = np.array(mono_audio.get_array_of_samples())
    max_val = float(1 << (8 * mono_audio.sample_width - 1))
    samples = samples.astype(np.float32) / max_val
    waveform = torch.from_numpy(samples).unsqueeze(0)

    audio_in_memory = {
        "waveform": waveform,
        "sample_rate": mono_audio.frame_rate
    }

    diarization = diarization_pipeline(audio_in_memory)
    print("      [Local] Diarization complete.")
    return diarization

def align_segments(whisper_data: dict, diarization) -> str:
    """Maps Whisper text segments to Pyannote speakers based on timestamp overlap."""
    print("  -> Aligning timestamps...")
    segments = whisper_data.get("segments", [])
    raw_transcript = []

    # Extract the actual Annotation object from the DiarizeOutput wrapper
    annotation = diarization.speaker_diarization if hasattr(diarization, "speaker_diarization") else diarization

    for segment in segments:
        seg_start = segment["start"]
        seg_end = segment["end"]
        seg_text = segment["text"].strip()

        # Calculate which speaker was active the longest during this text segment
        overlap_durations = {}
        for turn, _, speaker in annotation.itertracks(yield_label=True):
            overlap_start = max(seg_start, turn.start)
            overlap_end = min(seg_end, turn.end)
            overlap = max(0, overlap_end - overlap_start)

            if overlap > 0:
                overlap_durations[speaker] = overlap_durations.get(speaker, 0) + overlap

        # Default to UNKNOWN if no speaker track overlaps
        dominant_speaker = "UNKNOWN"
        if overlap_durations:
            dominant_speaker = max(overlap_durations, key=overlap_durations.get)

        raw_transcript.append(f"[{seg_start:.1f}s - {seg_end:.1f}s] {dominant_speaker}: {seg_text}")

    return "\n".join(raw_transcript)

def clean_transcript_with_llm(raw_transcript: str) -> str:
    """Uses Qwen3.5-397B-A17B-GPTQ-Int4 hosted by TU Wien's dataLAB to fix speaker errors based on conversational logic."""
    print("  -> Sending raw transcript to LLM for contextual cleanup...")

    system_prompt = (
        "You are an expert qualitative data assistant processing a user study transcript. "
        "The transcript format is '[start - end] SPEAKER: text'. "
        "The audio consists of a Think-Aloud protocol followed by an instructor interview. "
        "The automated speaker diarization has some errors (e.g., confusing speakers). "
        "Your task: "
        "1. Identify who is the 'Instructor' and who is the 'Student' based on the conversational context. "
        "2. Fix any obvious speaker misassignments where the diarization model failed. "
        "3. Output a clean, continuous transcript formatted clearly as 'Instructor: [text]' and 'Student: [text]'. "
        "4. DO NOT summarize, hallucinate, or remove any spoken words. Preserve the original meaning exactly."
    )

    response = client.chat.completions.create(
        model="qwen-3.5-397b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": raw_transcript}
        ],
        temperature=0.1 # Low temperature for strict adherence to the text
    )

    return response.choices[0].message.content.strip()

# Processing

In [8]:
executor = ThreadPoolExecutor(max_workers=2) # For parallel processing

for timeslot_folder in BASE_FOLDER.glob("Timeslot #*"):
    print(f"\n========================================")
    print(f"Processing: {timeslot_folder.name}")

    audio_dir = timeslot_folder / "audio"
    transcript_dir = timeslot_folder / "transcript"
    transcript_dir.mkdir(parents=True, exist_ok=True)

    audio_files = list(audio_dir.glob("*.*"))
    if not audio_files:
        print(f"  -> No audio file found in {audio_dir}. Skipping.")
        continue

    main_audio_path = audio_files[0]
    final_output_path = transcript_dir / f"{timeslot_folder.name}_final.txt"
    raw_output_path = transcript_dir / f"{timeslot_folder.name}_raw_aligned.txt"

    if final_output_path.exists():
        print(f"  -> Final transcript already exists. Skipping.")
        continue

    print(f"  -> Submitting tasks for Parallel Processing...")

    # Run API transcription and local diarization simultaneously
    future_transcription = executor.submit(transcribe_full_audio, main_audio_path)
    future_diarization = executor.submit(diarize_audio_locally, main_audio_path)

    # Wait for both tasks to complete and get their results
    whisper_data = future_transcription.result()
    diarization_data = future_diarization.result()

    # Align text with Speakers
    raw_aligned_text = align_segments(whisper_data, diarization_data)

    # Save the raw aligned text
    with open(raw_output_path, "w", encoding="utf-8") as f:
        f.write(raw_aligned_text)

    # LLM Contextual Cleanup
    final_transcript = clean_transcript_with_llm(raw_aligned_text)

    print(f"  -> Saving final transcript to {final_output_path.name}...")
    with open(final_output_path, "w", encoding="utf-8") as f:
        f.write(f"--- Final Cleaned Transcript for {timeslot_folder.name} ---\n\n")
        f.write(final_transcript)

print("\nAll user study folders processed successfully!")


Processing: Timeslot #0
  -> Final transcript already exists. Skipping.

All user study folders processed successfully!
